<a href="https://colab.research.google.com/github/24f2002727/Machine-learning-models/blob/main/SA-IIT-Guwhati-Hackathon-Converdion-prediction-model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Summer Analytics IIT Guwhati  Week 2 Hackathon
A machine learning model to predict the conversion of customers

In [ ]:
#import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer

from sklearn.preprocessing import LabelEncoder,OneHotEncoder,StandardScaler
from sklearn.metrics  import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix


In [ ]:
#load drive fpr dataset
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
train = pd.read_csv("/content/drive/MyDrive/Copy of train.csv")
test = pd.read_csv("/content/drive/MyDrive/Copy of public_test.csv")
private_test = pd.read_csv("/content/drive/MyDrive/Copy of private_test.csv")


In [ ]:
train.head()

,User_ID,Age,Income,City_Tier,Device_Type,Traffic_Source,Pages_Viewed,Products_Viewed,Time_On_Site,Previous_Purchases,Discount_Seen,Browser_Version,Campaign_Code,Converted
0,1,58.0,103593.708812,2,Mobile,Organic,5,4,9.61,3,0,11,2418,0
1,2,26.0,36451.716984,2,Mobile,Social Media,11,3,17.63,2,0,14,1213,0
2,3,19.0,30511.228700,3,Mobile,Referral,1,1,13.25,5,0,5,2849,0
3,4,48.0,87789.172342,3,Mobile,Email,14,12,NaN,1,1,19,7610,0
4,5,35.0,105229.249067,2,Mobile,Social Media,14,21,16.92,1,0,5,9261,0


In [ ]:
train.info()
test.info()
private_test.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   User_ID             10000 non-null  int64  
 1   Age                 8520 non-null   float64
 2   Income              9016 non-null   float64
 3   City_Tier           10000 non-null  int64  
 4   Device_Type         10000 non-null  object 
 5   Traffic_Source      10000 non-null  object 
 6   Pages_Viewed        10000 non-null  int64  
 7   Products_Viewed     10000 non-null  int64  
 8   Time_On_Site        8152 non-null   float64
 9   Previous_Purchases  10000 non-null  int64  
 10  Discount_Seen       10000 non-null  int64  
 11  Browser_Version     10000 non-null  int64  
 12  Campaign_Code       10000 non-null  int64  
 13  Converted           10000 non-null  int64  
dtypes: float64(3), int64(9), object(2)
memory usage: 1.1+ MB
<class 'pandas.core.frame.DataFrame'>
RangeInd

In [ ]:
#handle missing values
from sklearn.impute import SimpleImputer

print(train[['Income','Time_On_Site','Age']].describe())

# sns.boxplot(x=train['Income'])
# plt.show()
# sns.boxplot(x=train['Time_On_Site'])
# plt.show()
# sns.boxplot(x=train['Age'])
# plt.show()


income_imputer = SimpleImputer(strategy='mean')
median_imputer = SimpleImputer(strategy='median')

# Income
train[['Income']] = income_imputer.fit_transform(train[['Income']])
test[['Income']] = income_imputer.transform(test[['Income']])
private_test[['Income']] = income_imputer.transform(private_test[['Income']])

# Age and Time_On_Site
train[['Age', 'Time_On_Site']] = median_imputer.fit_transform(
    train[['Age', 'Time_On_Site']]
)

test[['Age', 'Time_On_Site']] = median_imputer.transform(
    test[['Age', 'Time_On_Site']]
)

private_test[['Age', 'Time_On_Site']] = median_imputer.transform(
    private_test[['Age', 'Time_On_Site']]
)

# sns.boxplot(x=train['Income'])
# plt.show()
# sns.boxplot(x=train['Time_On_Site'])
# plt.show()
# sns.boxplot(x=train['Age'])
# plt.show()

train.info()
test.info()
private_test.info()

              Income  Time_On_Site          Age
count    9016.000000   8152.000000  8520.000000
mean    69961.772797     13.667241    41.457746
std     24790.673822     19.438244    13.770164
min     12000.000000      0.800000    18.000000
25%     52294.644359      7.640000    29.000000
50%     70171.613672     11.145000    41.000000
75%     86907.747154     15.630000    53.000000
max    161687.774167    607.390000    65.000000
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   User_ID             10000 non-null  int64  
 1   Age                 10000 non-null  float64
 2   Income              10000 non-null  float64
 3   City_Tier           10000 non-null  int64  
 4   Device_Type         10000 non-null  object 
 5   Traffic_Source      10000 non-null  object 
 6   Pages_Viewed        10000 non-null  int64  
 7   Products_Viewe

In [ ]:


# #bining - data classifying into groups age,income
train['Age_group']=pd.cut(train['Age'],bins=3,labels=['Young','Adult','Old'])
test['Age_group']=pd.cut(test['Age'],bins=3,labels=['Young','Adult','Old'])
private_test['Age_group']=pd.cut(private_test['Age'],bins=3,labels=['Young','Adult','Old'])

train['Income_group']=pd.cut(train['Income'],bins=3,labels=['Low','Medium','High'])
test['Income_group']=pd.cut(test['Income'],bins=3,labels=['Low','Medium','High'])
private_test['Income_group']=pd.cut(private_test['Income'],bins=3,labels=['Low','Medium','High'])


#label encoding
# encoder=LabelEncoder()
# train['City_Tier']=encoder.fit_transform(train['City_Tier'])
# train['Device_Type']=encoder.fit_transform(train['Device_Type'])
# train['Traffic_Source']=encoder.fit_transform(train['Traffic_Source'])
# train['Age_group']=encoder.fit_transform(train['Age_group'])
# train['Income_group']=encoder.fit_transform(train['Income_group'])

# test['City_Tier']=encoder.fit_transform(test['City_Tier'])
# test['Device_Type']=encoder.fit_transform(test['Device_Type'])
# test['Traffic_Source']=encoder.transform(test['Traffic_Source'])
# test['Age_group']=encoder.fit_transform(test['Age_group'])
# test['Income_group']=encoder.fit_transform(test['Income_group'])

# private_test['City_Tier']=encoder.fit_transform(private_test['City_Tier'])
# private_test['Device_Type']=encoder.fit_transform(private_test['Device_Type'])
# private_test['Traffic_Source']=encoder.fit_transform(private_test['Traffic_Source'])
# private_test['Age_group']=encoder.fit_transform(private_test['Age_group'])
# private_test['Income_group']=encoder.fit_transform(private_test['Income_group'])

categorical_cols = [
    'City_Tier',
    'Device_Type',
    'Traffic_Source',
    'Age_group',
    'Income_group'
]

train = pd.get_dummies(
    train,
    columns=categorical_cols,
    drop_first=True
)

test = pd.get_dummies(
    test,
    columns=categorical_cols,
    drop_first=True
)

private_test = pd.get_dummies(
    private_test,
    columns=categorical_cols,
    drop_first=True
)

# Ensure test/private_test have same columns as train
test = test.reindex(columns=train.columns, fill_value=0)

private_test = private_test.reindex(
    columns=train.columns,
    fill_value=0
)

# train.info()




### Step 3: Define Features and Target

We separate the target column (`Converted`) from the rest of the features in the training set.
For simplicity, we only keep **numeric columns** in this baseline — categorical columns are dropped for now.
This keeps things easy to understand before we add more advanced preprocessing.


In [ ]:
TARGET = "Converted"

# Separate features and target
X_train = train.drop(columns=[TARGET])
y_train = train[TARGET]

X_test = test.drop(columns=[TARGET])
y_test = test[TARGET]

# X_train = train.drop(columns=[TARGET])
# y_train = train[TARGET]

# Keep only numeric columns for this baseline
numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()

X_train = X_train[numeric_cols]
X_test=X_test[numeric_cols]
X_private = private_test[numeric_cols]

X_test.info()
# X.private_test.info()

print("Features used:", numeric_cols)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   User_ID             3000 non-null   int64  
 1   Age                 3000 non-null   float64
 2   Income              3000 non-null   float64
 3   Pages_Viewed        3000 non-null   int64  
 4   Products_Viewed     3000 non-null   int64  
 5   Time_On_Site        3000 non-null   float64
 6   Previous_Purchases  3000 non-null   int64  
 7   Discount_Seen       3000 non-null   int64  
 8   Browser_Version     3000 non-null   int64  
 9   Campaign_Code       3000 non-null   int64  
dtypes: float64(3), int64(7)
memory usage: 234.5 KB
Features used: ['User_ID', 'Age', 'Income', 'Pages_Viewed', 'Products_Viewed', 'Time_On_Site', 'Previous_Purchases', 'Discount_Seen', 'Browser_Version', 'Campaign_Code']


In [ ]:
# Scale features to zero mean and unit variance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_private_scaled = scaler.transform(X_private)

print("Scaling done! Training data shape:", X_train_scaled.shape)


Scaling done! Training data shape: (10000, 10)


### Step 5: Train a Logistic Regression Model

Logistic Regression is one of the simplest and most interpretable classification algorithms — a great starting point for any binary classification problem.
It models the probability that a given input belongs to a class using a linear combination of features passed through a sigmoid function.
We set `max_iter=1000` to ensure the solver has enough iterations to converge on this dataset.


In [ ]:
model = LogisticRegression(max_iter=1000,class_weight='balanced',random_state=42)
model.fit(X_train_scaled, y_train)

predictions=model.predict(X_test_scaled)

print("Model trained successfully!")
print("Training Accuracy: {:.4f}".format(model.score(X_train, y_train)))




Model trained successfully!
Training Accuracy: 0.3087


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


In [ ]:
#evaluation of model

accr_score = accuracy_score(y_test,predictions)
prc_score=precision_score(y_test, predictions)
recl_score=recall_score(y_test, predictions)
f1score=f1_score(y_test,predictions)
cm = confusion_matrix(y_test, predictions)


print(accr_score)
print(prc_score)
print(recl_score)
print(f1score)
print(cm)

#accuracy_scores  : 0.7256666666666667
# f1_score  : 0.4040550325850833
#[[1898  216]
#  [ 607  279]]

# 0.725
# 0.5616161616161616
# 0.31376975169300225
# 0.40260680666183923
# [[1897  217]
#  [ 608  278]]

0.693
0.48315688161693937
0.5665914221218962
0.5215584415584416
[[1577  537]
 [ 384  502]]


In [ ]:
#prediction generation

predictions = model.predict(X_private_scaled)

submission = pd.DataFrame({
    "User_ID": private_test["User_ID"],
    "Converted": predictions
})

submission.to_csv("submission.csv", index=False)

print("submission.csv created successfully!")
submission.head()


submission.csv created successfully!


,User_ID,Converted
0,103001,0
1,103002,0
2,103003,0
3,103004,1
4,103005,0
